# LangChain: Memory 记忆管理

## 大纲
* ConversationBufferMemory 保存所有历史信息塞入promp
* ConversationBufferWindowMemory 保存最近n轮次会话
* ConversationTokenBufferMemory 保存最近n个字符大小的历史
* ConversationSummaryMemory 让llm为所有历史总结摘要

## 其他

除了大纲的四种管理历史会话的方式，langchain还支持向量数据存储，向量数据库存储所有的词嵌入embedding

In [ ]:
# 初始化
from langchain.chat_models import ChatOpenAI
import os
API_SECRET_KEY = "sk-scia8t0NThqBoFbDdzweXuPRmeQrMuQ9XkJaYl29VHyMACFD"
BASE_URL = "https://api.chatanywhere.tech"
os.environ["OPENAI_API_KEY"] = API_SECRET_KEY
os.environ["OPENAI_API_BASE"] = BASE_URL
llm_model = "gpt-3.5-turbo"

## ConversationBufferMemory 会话缓存记忆

用来保存会话上下文的，不然不记得讲过什么了

其实本质就是所有历史会话保存在prompt中，每次都把历史说过的所有会话发送过去

In [ ]:

# 忽视警告信息
import warnings
warnings.filterwarnings('ignore')
# 导入记忆管理组件和会话组件
from langchain.chains import ConversationChain
from langchain.memory import ConversationBufferMemory

llm = ChatOpenAI(temperature=0.0, model=llm_model)
memory = ConversationBufferMemory()

# 会话信息
conversation = ConversationChain(
    llm=llm, 
    memory = memory,
    verbose=True   # True会打印每次langchain格式化给llm的提示词
)

In [5]:
print(conversation.predict(input="Hi, my name is Andrew"))
print(conversation.predict(input="What is 1+1?"))
print(conversation.predict(input="What is my name?"))



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:

Human: Hi, my name is Andrew
AI:

> Finished chain.
Hello Andrew! It's nice to meet you. How can I assist you today?


> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
Human: Hi, my name is Andrew
AI: Hello Andrew! It's nice to meet you. How can I assist you today?
Human: What is 1+1?
AI:

> Finished chain.
The answer to 1+1 is 2.


> Entering new ConversationChain chain...
Prompt after formatting:
Th

In [6]:
print(memory.buffer)
memory.load_memory_variables({})

Human: Hi, my name is Andrew
AI: Hello Andrew! It's nice to meet you. How can I assist you today?
Human: What is 1+1?
AI: The answer to 1+1 is 2.
Human: What is my name?
AI: Your name is Andrew, as you mentioned earlier.


{'history': "Human: Hi, my name is Andrew\nAI: Hello Andrew! It's nice to meet you. How can I assist you today?\nHuman: What is 1+1?\nAI: The answer to 1+1 is 2.\nHuman: What is my name?\nAI: Your name is Andrew, as you mentioned earlier."}

In [9]:
# 手动构造一个历史会话
memory = ConversationBufferMemory()
memory.save_context({"input": "Hi"}, 
                    {"output": "What's up"})
print(memory.buffer)
memory.load_memory_variables({})

Human: Hi
AI: What's up


{'history': "Human: Hi\nAI: What's up"}

In [ ]:
# 继续添加会话
memory.save_context({"input": "Not much, just hanging"}, 
                    {"output": "Cool"})
memory.load_memory_variables({})

{'history': "Human: Hi\nAI: What's up\nHuman: Not much, just hanging\nAI: Cool"}

## ConversationBufferWindowMemory  会话缓存窗口记忆管理

随着会话进行，缓存越来越大，prompt历史信息越来越多，每次发送的token越来越长

给定一个窗口，prompt只保存过去窗口大小的会话历史

In [ ]:
from langchain.memory import ConversationBufferWindowMemory

# 创建带窗口的会话缓存
memory = ConversationBufferWindowMemory(k=1) # k=1，窗口大小是1，仅仅保存上一轮对话              
memory.save_context({"input": "Hi"},
                    {"output": "What's up"})
memory.save_context({"input": "Not much, just hanging"},
                    {"output": "Cool"})

memory.load_memory_variables({})

{'history': 'Human: Not much, just hanging\nAI: Cool'}

In [12]:
llm = ChatOpenAI(temperature=0.0, model=llm_model)
memory = ConversationBufferWindowMemory(k=1)
conversation = ConversationChain(
    llm=llm,            # 给定大模型
    memory = memory, # 给定会话历史
    verbose=False   # 不打印prompt
)

In [ ]:
# 这时候就不记得我的名字了
print(conversation.predict(input="Hi, my name is Andrew"))
print(conversation.predict(input="What is 1+1?"))
print(conversation.predict(input="What is my name?"))

"I'm sorry, I do not have access to your personal information, so I do not know your name."

## ConversationTokenBufferMemory 会话缓存字符大小内存管理

不再根据会话的轮数，而是根据历史n个字符，限制在历史大小为n个字符以内

In [14]:
#!pip install tiktoken

from langchain.memory import ConversationTokenBufferMemory
from langchain.llms import OpenAI
llm = ChatOpenAI(temperature=0.0, model=llm_model)

In [15]:
memory = ConversationTokenBufferMemory(llm=llm, max_token_limit=50)
memory.save_context({"input": "AI is what?!"},
                    {"output": "Amazing!"})
memory.save_context({"input": "Backpropagation is what?"},
                    {"output": "Beautiful!"})
memory.save_context({"input": "Chatbots are what?"}, 
                    {"output": "Charming!"})

In [18]:
memory.load_memory_variables({})

{'history': 'AI: Amazing!\nHuman: Backpropagation is what?\nAI: Beautiful!\nHuman: Chatbots are what?\nAI: Charming!'}

## ConversationSummaryMemory  会话缓存历史摘要记忆管理

让llm帮我们总结历史会话的摘要，减少会话缓存大小

In [ ]:
from langchain.memory import ConversationSummaryBufferMemory
# 创建一个很长的会话
schedule = "There is a meeting at 8am with your product team. \
You will need your powerpoint presentation prepared. \
9am-12pm have time to work on your LangChain \
project which will go quickly because Langchain is such a powerful tool. \
At Noon, lunch at the italian resturant with a customer who is driving \
from over an hour away to meet you to understand the latest in AI. \
Be sure to bring your laptop to show the latest LLM demo."

# 每次save_context上下文的时候，会判断是否超过100，超过会自动调用llm总结历史
memory = ConversationSummaryBufferMemory(llm=llm, max_token_limit=100)
memory.save_context({"input": "Hello"}, {"output": "What's up"})
memory.save_context({"input": "Not much, just hanging"},
                    {"output": "Cool"})
memory.save_context({"input": "What is on the schedule today?"}, 
                    {"output": f"{schedule}"})
memory.load_memory_variables({})

{'history': 'System: The human and AI exchange greetings and casual conversation. The AI provides a detailed schedule for the day, including a meeting with the product team, work on the LangChain project, and a lunch meeting with a customer interested in AI demo.'}

In [20]:
conversation = ConversationChain(
    llm=llm, 
    memory = memory,
    verbose=True
)

In [ ]:
# 在predict的时候，如果历史长度超过了限制，会再次调用llm总结历史
conversation.predict(input="What would be a good demo to show?")



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
System: The human and AI exchange greetings and casual conversation. The AI provides a detailed schedule for the day, including a meeting with the product team, work on the LangChain project, and a lunch meeting with a customer interested in AI demo.
Human: What would be a good demo to show?
AI:

> Finished chain.


'For the AI demo with the customer, a good idea would be to showcase some of the latest features and capabilities of our AI technology. This could include demonstrating natural language processing abilities, image recognition capabilities, and even showcasing some personalized recommendations based on user behavior. Additionally, we could show how the AI can automate certain tasks or streamline processes to improve efficiency. Would you like me to prepare a specific demo presentation for the meeting?'

In [22]:
memory.load_memory_variables({})

{'history': 'System: The human and AI exchange greetings and casual conversation. The AI provides a detailed schedule for the day, including a meeting with the product team, work on the LangChain project, and a lunch meeting with a customer interested in AI demo. The human asks what would be a good demo to show.\nAI: For the AI demo with the customer, a good idea would be to showcase some of the latest features and capabilities of our AI technology. This could include demonstrating natural language processing abilities, image recognition capabilities, and even showcasing some personalized recommendations based on user behavior. Additionally, we could show how the AI can automate certain tasks or streamline processes to improve efficiency. Would you like me to prepare a specific demo presentation for the meeting?'}

Reminder: Download your notebook to you local computer to save your work.